# Sound Files, Headers and Formats

Everything about reading and writing sound files: the `mus_sound_*` API, header and sample types, and format round-trips.

Docs: [file IO](https://testcase.github.io/pysndlib/file_IO.html) | [sndlib functions](https://testcase.github.io/pysndlib/sndlib_functions.html)

In [ ]:
import os, importlib
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

import pysndlib.clm as clm
import pysndlib.sndlib as sndlib

SR = 44100

# make a stereo test sound
with clm.Sound('files_demo.wav', channels=2, srate=SR, comment='pysndlib files demo'):
    osc1 = clm.make_oscil(440.0)
    osc2 = clm.make_oscil(660.0)
    for i in range(SR):
        clm.outa(i, 0.3 * clm.oscil(osc1))
        clm.outb(i, 0.3 * clm.oscil(osc2))

## The mus_sound_* API

In [ ]:
print('samples   :', sndlib.mus_sound_samples('files_demo.wav'))    # frames * chans
print('framples  :', sndlib.mus_sound_framples('files_demo.wav'))  # frames (per channel)
print('srate     :', sndlib.mus_sound_srate('files_demo.wav'))
print('chans     :', sndlib.mus_sound_chans('files_demo.wav'))
print('duration  :', sndlib.mus_sound_duration('files_demo.wav'))
print('header    :', sndlib.mus_header_type_to_string(sndlib.mus_sound_header_type('files_demo.wav')))
print('sample type:', sndlib.mus_sample_type_to_string(sndlib.mus_sound_sample_type('files_demo.wav')))
print('comment   :', sndlib.mus_sound_comment('files_demo.wav'))

`samples` counts every sample across channels; `framples` is frames (samples per channel) -- the number you iterate per channel.

## Reading data

In [ ]:
data, sr = clm.file2ndarray('files_demo.wav')
print('array shape:', data.shape, '(chans, framples)')
plt.figure(figsize=(14, 2.5))
plt.plot(data[0][:2000], label='chan 0 (440)')
plt.plot(data[1][:2000], label='chan 1 (660)', alpha=0.7)
plt.legend(); plt.show()

## Writing ndarrays

`ndarray2file` writes any (chans, framples) array back out, choosing header and sample type:

In [ ]:
louder = data * 1.5
clm.ndarray2file('files_loud.wav', louder, sr=SR)
d2, _ = clm.file2ndarray('files_loud.wav')
print('round trip shape:', d2.shape)

In [ ]:
ipd.Audio('files_loud.wav', rate=SR)

## Formats

pysndlib reads and writes many header and sample formats (RIFF, AIFF, NeXT/Sun, and more; float or integer sample types). Round-trip the same data through AIFF and check what changed:

In [ ]:
for hdr in [sndlib.Header.RIFF, sndlib.Header.AIFF, sndlib.Header.NEXT]:
    name = f'files_{int(hdr)}.aiff' if hdr == sndlib.Header.AIFF else f'files_{int(hdr)}.wav'
    try:
        with clm.Sound(name, channels=2, srate=SR, header_type=hdr):
            osc1 = clm.make_oscil(440.0); osc2 = clm.make_oscil(660.0)
            for i in range(SR // 2):
                clm.outa(i, 0.3 * clm.oscil(osc1)); clm.outb(i, 0.3 * clm.oscil(osc2))
        print('%-22s header=%-8s samples=%d srate=%d' % (
            name,
            sndlib.mus_header_type_to_string(sndlib.mus_sound_header_type(name)),
            sndlib.mus_sound_samples(name), sndlib.mus_sound_srate(name)))
    except Exception as e:
        print(name, '->', e)

## The shipped examples

In [ ]:
examples_dir = os.path.dirname(importlib.import_module('pysndlib.examples').__file__)
for f in sorted(os.listdir(examples_dir)):
    if f.endswith(('.wav', '.snd', '.aiff', '.aif')):
        p = os.path.join(examples_dir, f)
        try:
            print('%-22s chans=%d srate=%6d framples=%7d' % (
                f, sndlib.mus_sound_chans(p), sndlib.mus_sound_srate(p),
                sndlib.mus_sound_framples(p)))
        except Exception as e:
            print(f, '->', e)

## Summary

- `mus_sound_*` reads headers/metadata without loading data; `file2ndarray`/`ndarray2file` move the data
- header/sample types are first-class: mix formats freely

Next: [spatialization and reverb](16_Spatialization_Reverb.ipynb)